# Can we predict a warranty claim within a year?
Protocol: `protocol.yaml`. Holdout rows from `holdout.csv`; 5-fold stratified CV on training rows only; model chosen by highest mean CV ROC AUC; holdout scored once.

Features: `channel`, `device_age_years`, `price_eur`, `daily_hours` (all known at sale/registration). `device_serial` excluded (identifier).

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

SEED = 0
sha = lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert sha('data.csv') == '25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
assert sha('holdout.csv') == '083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==7000 and len(hold_idx)==1400 and len(set(hold_idx))==1400
train_idx = np.setdiff1d(np.arange(len(df)), hold_idx)
TARGET='claim_within_year'
CAT=['channel']; NUM=['device_age_years','price_eur','daily_hours']
FEATURES=CAT+NUM
X, y = df[FEATURES], df[TARGET]
Xtr, ytr = X.iloc[train_idx], y.iloc[train_idx]
Xho, yho = X.iloc[hold_idx], y.iloc[hold_idx]
print(len(train_idx), len(hold_idx), ytr.mean(), yho.mean())

In [ ]:
def prep(scale):
    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT),
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))] + ([('sc', StandardScaler())] if scale else [])), NUM)])
# All candidates named before any is scored; preprocessing lives inside the pipeline so it is fit per CV fold.
candidates = {
  'baseline_prior': Pipeline([('p', prep(False)), ('m', DummyClassifier(strategy='prior'))]),
  'logreg_C0.1': Pipeline([('p', prep(True)), ('m', LogisticRegression(C=0.1, max_iter=2000))]),
  'logreg_C1': Pipeline([('p', prep(True)), ('m', LogisticRegression(C=1.0, max_iter=2000))]),
  'rf_depth5': Pipeline([('p', prep(False)), ('m', RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=20, random_state=SEED, n_jobs=-1))]),
  'hgb_small': Pipeline([('p', prep(False)), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=SEED))]),
}
print(list(candidates))

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(Xtr, ytr))
cv_scores = {}
for name, est in candidates.items():
    s=[]
    for tr, va in folds:
        m = clone(est).fit(Xtr.iloc[tr], ytr.iloc[tr])
        s.append(float(roc_auc_score(ytr.iloc[va], m.predict_proba(Xtr.iloc[va])[:,1])))
    cv_scores[name]=s
cv = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cv.sort_values('mean', ascending=False)

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen, np.mean(cv_scores[chosen]))
final = clone(candidates[chosen]).fit(Xtr, ytr)
holdout_auc = float(roc_auc_score(yho, final.predict_proba(Xho)[:,1]))   # scored once
print('holdout roc_auc:', holdout_auc)

In [ ]:
evidence = {
  'features': FEATURES,
  'candidates': list(candidates),
  'cv_folds': [[train_idx[tr].tolist(), train_idx[va].tolist()] for tr,va in folds],
  'cv_scores': cv_scores,
  'chosen': chosen,
  'holdout_rows': hold_idx.tolist(),
  'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))